# Explore Sales Orders

This notebook runs the same processing as the scripts in `src/`, from loading the raw CSV through the final charts. It does the whole job in memory, so it never reads or writes `data/sales_clean.csv`. Run the cells in order. The notebook keeps its own session, so its variables stay separate from the Console's.

## Load the Raw Data

Import the project's libraries and read `data/sales_data.csv`. The kernel starts in this `notebooks/` folder, so `PROJECT_ROOT` steps up one level to reach the project root.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().parent

sales = pd.read_csv(PROJECT_ROOT / "data/sales_data.csv")

print(sales.shape)
print(sales.head())

## Check Data Quality

Count missing cells, duplicate rows, and repeated order numbers, then review the column types.

In [ ]:
print("Missing cells:", sales.isna().sum().sum())
print("Duplicate rows:", sales.duplicated().sum())
print(
    "Repeated order numbers:",
    sales["order_number"].duplicated().sum(),
)
print(sales.dtypes)

## Clean the Data

Fix the misspelled `produce_name` column, parse `order_date` as datetimes, and normalize the category labels to title case. The analysis below works from `clean_sales` directly.

In [ ]:
clean_sales = sales.rename(columns={"produce_name": "product_name"}).copy()
clean_sales["order_date"] = pd.to_datetime(
    clean_sales["order_date"], format="%d/%m/%Y"
)
clean_sales["product_category"] = (
    clean_sales["product_category"].str.strip().str.title()
)

## Describe the Data

Compare mean and median order values, and check how many retail and wholesale orders you have.

In [ ]:
print(
    clean_sales[["quantity", "unit_price", "sale_price"]].describe().round(2)
)
print(clean_sales["order_type"].value_counts())

## Compare Typical Orders

Calculate the median order value within each product category and order type. Keep group counts and quantity and unit-price medians for context.

In [ ]:
order_summary = clean_sales.groupby(
    ["product_category", "order_type"], as_index=False
).agg(
    orders=("order_number", "size"),
    median_order_value=("sale_price", "median"),
    median_quantity=("quantity", "median"),
    median_unit_price=("unit_price", "median"),
)
print(order_summary.round(2).to_string(index=False))

## Visualize the Comparison

Compare the retail and wholesale bars within each category. These figures describe a fictional dataset.

In [ ]:
chart_data = order_summary.pivot(
    index="product_category",
    columns="order_type",
    values="median_order_value",
)
ax = chart_data.plot.bar(rot=0, figsize=(8, 4))
ax.set(xlabel="Product category", ylabel="Median order value")
ax.set_title("Retail and Wholesale Order Values")
plt.tight_layout()
plt.show()

## Optional Assistant Follow-Up

Ask whether larger quantities help explain the retail/wholesale differences. Check unit prices and product mix as well. Group medians alone do not establish a cause, and multiplying medians does not generally give the median of a product. The starting calculation to verify is already in `order_summary`.